# ADMET-Engine v3.0 (*CardioPhys-3D*) — WP3 GPU Training Notebook (Colab Pro A100 / L4 / T4)

Trains the **SE(3)-equivariant PaiNN 3D protein-ligand model** (`CardioPhys3DPredictor`) on the WP1 multi-channel ChEMBL v37 dataset (**hERG**, **Cav1.2**, **Nav1.5**) with:
- Censored Gaussian **Tobit Negative Log-Likelihood** (`=`, `>`, `<`)
- **bfloat16 / float16** mixed precision on CUDA GPU
- **Boltzmann conformer sampling** at 310.15 K
- **Mondrian Conformal Prediction** (90% finite-sample coverage guarantee)
- Automatic Google Drive restore (from WP1) & backup (for WP4)

> **Want to run WP1 → WP5 in one notebook without switching runtimes?** Use `notebooks/cardiophys_3d_master_all_in_one.ipynb`!

In [ ]:
# @title 1. Sync Repo, Install Dependencies & Verify GPU
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/content/Admet-Engine')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/JakobFerschin/Admet-Engine.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'rdkit', 'torch-geometric', 'joblib', 'scikit-learn', 'scipy', 'requests', 'pandas', 'tqdm'],
    check=True,
)
(REPO_DIR / 'data' / 'processed').mkdir(parents=True, exist_ok=True)
(REPO_DIR / 'models').mkdir(parents=True, exist_ok=True)

import torch
if not torch.cuda.is_available():
    print('WARNING: CUDA GPU not detected! Switch Runtime -> Change runtime type -> GPU for fast training.')
else:
    print('GPU:', torch.cuda.get_device_name(0), '| bfloat16:', torch.cuda.is_bf16_supported())

In [ ]:
# @title 2. Optional: Mount Google Drive & Restore WP1 Artifacts
import shutil
from pathlib import Path

ENABLE_DRIVE_BACKUP = True  # @param {type:'boolean'}
DRIVE_DIR = Path('/content/drive/MyDrive/CardioPhys3D_v3')
DRIVE_MOUNTED = False

DATA_CSV = Path('data/processed/cardiophys_multichannel_v3.csv')
CONF_PT = Path('data/processed/cardiophys_3d_conformers_v3.pt')

if ENABLE_DRIVE_BACKUP:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        DRIVE_MOUNTED = True
        restore_map = [
            (DATA_CSV, ['cardiophys_multichannel_v3.csv', 'v3_multichannel_chembl37.csv']),
            (CONF_PT, ['cardiophys_3d_conformers_v3.pt', 'v3_conformers_3d.pt']),
        ]
        for dest, candidates in restore_map:
            if not dest.exists():
                for fname in candidates:
                    src = DRIVE_DIR / fname
                    if src.exists():
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(src, dest)
                        print(f'Restored {fname} from Drive -> {dest}')
                        break
    except Exception as e:
        print(f'Google Drive mount skipped (non-fatal): {e}')

if not (DATA_CSV.exists() and CONF_PT.exists()):
    raise FileNotFoundError(
        f'Missing WP1 artifacts ({DATA_CSV} and/or {CONF_PT}). '
        'Run cardiophys_3d_wp1_wp2.ipynb first (with Drive backup enabled), '
        'or use the 1-Click Master Notebook: cardiophys_3d_master_all_in_one.ipynb.'
    )
print(f'WP1 artifacts ready: {DATA_CSV} & {CONF_PT}')

In [ ]:
# @title 3. Train SE(3)-Equivariant CardioPhys-3D with Tobit Censored NLL
import subprocess, sys

EPOCHS = 40       # @param {type:'integer'}
BATCH_SIZE = 64   # @param {type:'integer'}
HIDDEN_DIM = 128  # @param {type:'integer'}
NUM_LAYERS = 3    # @param {type:'integer'}
LR = 5e-4         # @param {type:'number'}

subprocess.run([
    sys.executable, 'src/training/train_wp3_3d.py',
    '--csv', str(DATA_CSV),
    '--conformers', str(CONF_PT),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--hidden-dim', str(HIDDEN_DIM),
    '--num-layers', str(NUM_LAYERS),
    '--lr', str(LR),
    '--out-dir', 'models',
], check=True)

In [ ]:
# @title 4. Inspect Held-Out Scaffold Metrics & Backup Teacher Checkpoint to Drive
import json, shutil
from pathlib import Path

metrics_path = Path('models/cardiophys_3d_v3_metrics.json')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(metrics, indent=2))

if DRIVE_MOUNTED:
    for p in (Path('models/cardiophys_3d_v3.pt'), metrics_path):
        if p.exists():
            shutil.copy2(p, DRIVE_DIR / p.name)
            print(f'Backed up {p.name} -> {DRIVE_DIR / p.name}')